<a href="https://colab.research.google.com/github/profliuhao/CSIT599/blob/main/CSIT599_Online_lab5_seq2seq_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/profliuhao/CSIT599/blob/main/CSIT599_Online_lab5_seq2seq_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 5 — Sequence-to-Sequence Translation With and Without Attention (Keras)

**Module 5: Attention Mechanisms and Transformers**

In this lab you will build **two English → German translators** in Keras and measure
exactly what attention buys you:

* **Part A — seq2seq WITHOUT attention:** an LSTM encoder squeezes the whole source
  sentence into one fixed-size state, and an LSTM decoder must translate from that alone.
* **Part B — seq2seq WITH attention:** a **Bahdanau attention** layer lets the decoder
  form a weighted view over *all* encoder states instead of one bottleneck vector.
* **Part C — comparison:** loss, masked token accuracy, BLEU, and sample translations.

## What you will do
Fill in each `___BLANK___` (a hint comment sits next to every one):
  1. **padding** choices in the data pipeline
  2. the **masked-accuracy** argmax axis
  3. key arguments of the **encoder** (embedding size, `return_sequences`, `return_state`)
  4. the decoder's **output dimension** and the optimizer's **gradient update**
  5. the heart of **Bahdanau attention**: the score layer, activation, and softmax axis

Everything else — the dataset download, tokenization, training loops, BLEU, and the
comparison — is already written for you.

## How to use this file
* Use **Google Colab with a GPU runtime** (`Runtime > Change runtime type > T4 GPU`).
  Training both models takes ~10-15 minutes on GPU.
* Fill in each `___BLANK___`, then run the cells from top to bottom (or "Run All").

## Setup — imports and hyperparameters

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import urllib.request
import zipfile
import os
import re
from tqdm.notebook import tqdm

np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")

In [ ]:
# ==============================================================================
# HYPERPARAMETERS
# ==============================================================================
BATCH_SIZE = 64        # Samples per gradient update
EMBEDDING_DIM = 512    # Dimension of the word embeddings
LSTM_UNITS = 256       # Hidden units in the LSTM layers (model capacity)
EPOCHS = 20            # Full passes through the training data
MAX_VOCAB_SIZE = 10000 # Keep only the most frequent words
MAX_LENGTH = 20        # Pad/truncate every sentence to this many tokens

## PART 0 — Data: download, preprocess, tokenize *(mostly provided)*

We use the **English-German sentence pairs** from manythings.org (Tatoeba project).
Every sentence is lowercased, punctuation is separated, and `<start>`/`<end>` tokens
are added so the decoder knows where sentences begin and end.

In [ ]:
def download_data():
    """Download the English-German translation dataset if not already present."""
    url = "http://www.manythings.org/anki/deu-eng.zip"
    filename = "deu-eng.zip"
    if not os.path.exists("deu.txt"):
        print("Downloading dataset...")
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3"}
        req = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(req) as response, open(filename, "wb") as out_file:
            out_file.write(response.read())
        with zipfile.ZipFile(filename, "r") as zip_ref:
            zip_ref.extractall()
        os.remove(filename)
        print("Download complete!")
    else:
        print("Dataset already exists.")

def preprocess_sentence(sentence):
    """Lowercase, add spaces around punctuation, and add <start>/<end> tokens."""
    sentence = sentence.lower().strip()
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)   # separate punctuation
    sentence = re.sub(r'[" "]+', " ", sentence)          # collapse multiple spaces
    sentence = sentence.strip()
    sentence = "<start> " + sentence + " <end>"
    return sentence

def load_dataset(num_examples=10000):
    """Load and preprocess English-German pairs from 'deu.txt'."""
    download_data()
    with open("deu.txt", "r", encoding="utf-8") as f:
        lines = f.read().strip().split("\n")
    pairs = []
    for line in lines[:num_examples]:
        parts = line.split("\t")
        if len(parts) >= 2:
            eng = preprocess_sentence(parts[0])
            deu = preprocess_sentence(parts[1])
            if len(eng.split()) <= MAX_LENGTH and len(deu.split()) <= MAX_LENGTH:
                pairs.append([eng, deu])
    print(f"Loaded {len(pairs)} sentence pairs")
    return zip(*pairs)

input_texts, target_texts = load_dataset(num_examples=20000)  # ~20k short sentence pairs
input_texts, target_texts = list(input_texts), list(target_texts)
print("Example pair:")
print("  EN:", input_texts[100])
print("  DE:", target_texts[100])

In [ ]:
# ==============================================================================
# TOKENIZATION: text sentences -> padded integer sequences
# ==============================================================================
input_tokenizer = keras.preprocessing.text.Tokenizer(
    num_words=MAX_VOCAB_SIZE,
    filters='!"#$%&()*+,-./:;=?@[\\]^_`{|}~\t\n',  # keep '<' and '>' for <start>/<end>
    oov_token="<UNK>",
)
target_tokenizer = keras.preprocessing.text.Tokenizer(
    num_words=MAX_VOCAB_SIZE,
    filters='!"#$%&()*+,-./:;=?@[\\]^_`{|}~\t\n',
    oov_token="<UNK>",
)

input_tokenizer.fit_on_texts(input_texts)
target_tokenizer.fit_on_texts(target_texts)

input_sequences = input_tokenizer.texts_to_sequences(input_texts)
target_sequences = target_tokenizer.texts_to_sequences(target_texts)

# ___BLANK___: What type of padding should we use? ('pre' or 'post')
# Hint: seq2seq models conventionally pad with zeros AT THE END of the sequence,
# so the LSTM reads the real words first.
input_sequences = keras.preprocessing.sequence.pad_sequences(
    input_sequences, maxlen=MAX_LENGTH,
    padding=___BLANK___)
target_sequences = keras.preprocessing.sequence.pad_sequences(
    target_sequences, maxlen=MAX_LENGTH,
    padding=___BLANK___)

# +1 because index 0 is reserved for padding
input_vocab_size = min(MAX_VOCAB_SIZE, len(input_tokenizer.word_index)) + 1
target_vocab_size = min(MAX_VOCAB_SIZE, len(target_tokenizer.word_index)) + 1
print(f"Input (English) vocabulary size: {input_vocab_size}")
print(f"Target (German) vocabulary size: {target_vocab_size}")

# 90/10 train/validation split
split_idx = int(0.9 * len(input_sequences))
train_input, val_input = input_sequences[:split_idx], input_sequences[split_idx:]
train_target, val_target = target_sequences[:split_idx], target_sequences[split_idx:]
print(f"Training pairs: {len(train_input)}  |  Validation pairs: {len(val_input)}")

## Evaluation metrics *(one blank)*

Plain accuracy would reward the model for "predicting" padding zeros, so we mask them
out. BLEU gives a rough measure of translation quality.

In [ ]:
def calculate_accuracy(predictions, targets):
    """Token-level accuracy that ignores padding tokens.

    predictions: [batch, seq_len, vocab_size] logits;  targets: [batch, seq_len] ids.
    """
    # ___BLANK___: Which axis do we argmax over to get predicted token ids?
    # Hint: the highest-scoring entry of the VOCABULARY dimension (the last one).
    predicted_ids = tf.argmax(predictions, axis=___BLANK___)
    mask = tf.math.not_equal(targets, 0)             # True where the target is NOT padding
    matches = tf.equal(predicted_ids, targets)
    matches = tf.boolean_mask(matches, mask)         # keep only non-padding positions
    return tf.reduce_mean(tf.cast(matches, tf.float32)).numpy()

def calculate_bleu_score(reference, candidate):
    """Very simple unigram-precision BLEU with a brevity penalty (0 = worst, 1 = perfect)."""
    reference_tokens = [t for t in reference.lower().split() if t not in ["<start>", "<end>"]]
    candidate_tokens = [t for t in candidate.lower().split() if t not in ["<start>", "<end>"]]
    if len(candidate_tokens) == 0:
        return 0.0
    matches = sum(1 for t in candidate_tokens if t in reference_tokens)
    precision = matches / len(candidate_tokens)
    brevity = min(1.0, len(candidate_tokens) / max(len(reference_tokens), 1))
    return precision * brevity

## PART A — Seq2Seq WITHOUT attention

### A.1 Encoder
The encoder reads the English sentence and compresses it into its **final LSTM states**
`(state_h, state_c)` — the fixed-size "context vector" the decoder must translate from.

In [ ]:
class Encoder_NoAttention(keras.Model):
    """LSTM encoder: input sentence -> final hidden/cell states (the context)."""
    def __init__(self, vocab_size, embedding_dim, lstm_units):
        super(Encoder_NoAttention, self).__init__()

        # ___BLANK___: What is the input dimension of the embedding layer?
        # Hint: one row per word id — the size of the INPUT (English) vocabulary.
        self.embedding = layers.Embedding(
            input_dim=___BLANK___,
            output_dim=embedding_dim,
            mask_zero=True)

        # ___BLANK___ x2: The LSTM must give us BOTH the per-timestep outputs AND the final states.
        # Hint: both arguments are booleans.
        self.lstm = layers.LSTM(
            lstm_units,
            return_sequences=___BLANK___,
            return_state=___BLANK___,
            name="encoder_lstm_no_attention")

    def call(self, x):
        x = self.embedding(x)                            # [batch, MAX_LENGTH, embedding_dim]
        encoder_outputs, state_h, state_c = self.lstm(x) # outputs + final states
        return encoder_outputs, state_h, state_c

### A.2 Decoder (no attention)
The decoder is another LSTM, initialized with the encoder's final states. At each step
it consumes the previous target word (teacher forcing during training) and predicts the
next one through a Dense layer over the German vocabulary.

In [ ]:
class Decoder_NoAttention(keras.Model):
    """LSTM decoder conditioned only on the encoder's final states."""
    def __init__(self, vocab_size, embedding_dim, lstm_units):
        super(Decoder_NoAttention, self).__init__()
        self.embedding = layers.Embedding(
            input_dim=vocab_size, output_dim=embedding_dim, mask_zero=True)
        self.lstm = layers.LSTM(
            lstm_units, return_sequences=True, return_state=True,
            name="decoder_lstm_no_attention")

        # ___BLANK___: What is the output dimension of the final Dense layer?
        # Hint: one score (logit) per word of the TARGET vocabulary.
        self.dense = layers.Dense(
            ___BLANK___)

    def call(self, x, initial_state):
        x = self.embedding(x)                                     # [batch, seq, emb]
        lstm_output, state_h, state_c = self.lstm(x, initial_state=initial_state)
        outputs = self.dense(lstm_output)                         # [batch, seq, vocab]
        return outputs

print("\nBuilding Seq2Seq model WITHOUT attention...")
encoder_no_attn = Encoder_NoAttention(input_vocab_size, EMBEDDING_DIM, LSTM_UNITS)
decoder_no_attn = Decoder_NoAttention(target_vocab_size, EMBEDDING_DIM, LSTM_UNITS)
print("✓ Encoder and Decoder (WITHOUT attention) created successfully")

### A.3 Training setup (no attention) *(one blank)*

Teacher forcing: the decoder input is the target shifted right (`<start> w1 w2 ...`)
and the loss compares its predictions to the target shifted left (`w1 w2 ... <end>`).
Padding positions are masked out of the loss with `sample_weight`.

In [ ]:
optimizer_no_attn = keras.optimizers.Adam(learning_rate=0.001, clipnorm=1.0)
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

def train_step_no_attention(input_batch, target_batch):
    """One training step: forward pass, masked loss, backward pass, update."""
    with tf.GradientTape() as tape:
        encoder_outputs, state_h, state_c = encoder_no_attn(input_batch)
        decoder_input = target_batch[:, :-1]    # <start> w1 w2 ...   (what the decoder sees)
        decoder_target = target_batch[:, 1:]    # w1 w2 ... <end>     (what it must predict)
        predictions = decoder_no_attn(decoder_input, [state_h, state_c])
        mask = tf.math.not_equal(decoder_target, 0)
        loss = loss_fn(decoder_target, predictions, sample_weight=mask)
        accuracy = calculate_accuracy(predictions, decoder_target)

    trainable_vars = encoder_no_attn.trainable_variables + decoder_no_attn.trainable_variables
    gradients = tape.gradient(loss, trainable_vars)
    # ___BLANK___: Which optimizer method applies the gradients to the variables?
    # Hint: it takes zip(gradients, variables).
    optimizer_no_attn.___BLANK___(zip(gradients, trainable_vars))
    return loss.numpy(), accuracy

def evaluate_no_attention(input_data, target_data):
    """Average masked loss/accuracy over a dataset (no gradient updates)."""
    total_loss = total_accuracy = 0
    num_batches = len(input_data) // BATCH_SIZE
    for i in range(num_batches):
        input_batch = input_data[i * BATCH_SIZE:(i + 1) * BATCH_SIZE]
        target_batch = target_data[i * BATCH_SIZE:(i + 1) * BATCH_SIZE]
        encoder_outputs, state_h, state_c = encoder_no_attn(input_batch)
        decoder_input, decoder_target = target_batch[:, :-1], target_batch[:, 1:]
        predictions = decoder_no_attn(decoder_input, [state_h, state_c])
        mask = tf.math.not_equal(decoder_target, 0)
        total_loss += loss_fn(decoder_target, predictions, sample_weight=mask).numpy()
        total_accuracy += calculate_accuracy(predictions, decoder_target)
    return total_loss / num_batches, total_accuracy / num_batches

### A.4 Train the model WITHOUT attention *(provided)*

In [ ]:
print("\n" + "="*70)
print("TRAINING SEQ2SEQ WITHOUT ATTENTION")
print("="*70)

best_val_loss_no_attn, best_val_acc_no_attn = float("inf"), 0

for epoch in tqdm(range(EPOCHS)):
    indices = np.random.permutation(len(train_input))
    train_input_shuffled = train_input[indices]
    train_target_shuffled = train_target[indices]

    num_batches = len(train_input) // BATCH_SIZE
    total_loss = total_accuracy = 0
    for i in range(num_batches):
        input_batch = train_input_shuffled[i * BATCH_SIZE:(i + 1) * BATCH_SIZE]
        target_batch = train_target_shuffled[i * BATCH_SIZE:(i + 1) * BATCH_SIZE]
        loss, accuracy = train_step_no_attention(input_batch, target_batch)
        total_loss += loss
        total_accuracy += accuracy

    train_loss, train_accuracy = total_loss / num_batches, total_accuracy / num_batches
    val_loss, val_accuracy = evaluate_no_attention(val_input, val_target)
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {train_loss:.4f}, Acc: {train_accuracy:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.4f}")
    if val_loss < best_val_loss_no_attn:
        best_val_loss_no_attn, best_val_acc_no_attn = val_loss, val_accuracy

print(f"\n✓ Training complete!")
print(f"  Best Validation Loss (NO ATTENTION): {best_val_loss_no_attn:.4f}")
print(f"  Best Validation Accuracy (NO ATTENTION): {best_val_acc_no_attn:.4f} ({best_val_acc_no_attn*100:.2f}%)")

## PART B — Seq2Seq WITH attention

### B.1 Encoder
Identical to before — but now the per-timestep outputs matter, because attention will
look at **all** of them, not just the final state.

In [ ]:
class Encoder_WithAttention(keras.Model):
    """LSTM encoder whose full output sequence feeds the attention mechanism."""
    def __init__(self, vocab_size, embedding_dim, lstm_units):
        super(Encoder_WithAttention, self).__init__()
        self.embedding = layers.Embedding(
            input_dim=vocab_size, output_dim=embedding_dim, mask_zero=True)
        self.lstm = layers.LSTM(
            lstm_units,
            return_sequences=True,  # MUST return all timesteps for attention
            return_state=True,
            name="encoder_lstm_with_attention")

    def call(self, x):
        x = self.embedding(x)
        encoder_outputs, state_h, state_c = self.lstm(x)
        return encoder_outputs, state_h, state_c

### B.2 Bahdanau (additive) attention — *the heart of this lab*

For a decoder state $s$ and encoder outputs $h_1 \dots h_T$:

$$\text{score}(s, h_i) = V^\top \tanh(W_1 h_i + W_2 s), \qquad
  \alpha = \text{softmax}(\text{score}), \qquad
  \text{context} = \sum_i \alpha_i h_i$$

The weights $\alpha$ say *which source words matter right now*; the context vector is
their weighted summary.

In [ ]:
class BahdanauAttention(keras.layers.Layer):
    """Additive attention: a learned, differentiable lookup over encoder outputs."""
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = layers.Dense(units)   # transforms encoder outputs
        self.W2 = layers.Dense(units)   # transforms the decoder state

        # ___BLANK___: What is the output dimension of V?
        # Hint: V turns each position's energy vector into a SINGLE alignment score.
        self.V = layers.Dense(___BLANK___)

    def call(self, decoder_hidden, encoder_outputs):
        # decoder_hidden: [batch, units];  encoder_outputs: [batch, MAX_LENGTH, units]
        decoder_hidden_with_time = tf.expand_dims(decoder_hidden, axis=1)  # [batch, 1, units]

        # ___BLANK___: Which activation does Bahdanau attention use for the energy?
        # Hint: the additive-attention formula uses the hyperbolic tangent.
        energy = tf.nn.___BLANK___(
            self.W1(encoder_outputs) + self.W2(decoder_hidden_with_time))

        attention_scores = self.V(energy)                     # [batch, MAX_LENGTH, 1]

        # ___BLANK___: Along which axis do we softmax so the weights over the
        # SOURCE POSITIONS sum to 1?  Hint: the sequence-length axis.
        attention_weights = tf.nn.softmax(attention_scores, axis=___BLANK___)

        context_vector = tf.reduce_sum(attention_weights * encoder_outputs, axis=1)
        return context_vector, attention_weights

### B.3 Decoder with attention *(provided — read it!)*

For training speed this decoder uses a **vectorized simplification**: it computes one
attention context from the encoder's final state and feeds it (tiled) to every decoder
timestep, instead of re-attending at each step. That keeps the full attention machinery
you implemented above (scores → softmax → context) while training in one pass; the
classic per-step version behaves the same way but is slower in a Python loop.

In [ ]:
class Decoder_WithAttention(keras.Model):
    """Decoder that concatenates an attention context with each input embedding."""
    def __init__(self, vocab_size, embedding_dim, lstm_units):
        super(Decoder_WithAttention, self).__init__()
        self.embedding = layers.Embedding(
            input_dim=vocab_size, output_dim=embedding_dim, mask_zero=True)
        self.attention = BahdanauAttention(lstm_units)
        self.lstm = layers.LSTM(lstm_units, return_sequences=True, return_state=True)
        self.dense = layers.Dense(vocab_size)

    def call(self, x, encoder_outputs, initial_state):
        x = self.embedding(x)                       # [batch, seq, emb]
        state_h, state_c = initial_state

        # Attention over ALL encoder outputs, queried by the encoder's final state.
        context_vector, attention_weights = self.attention(state_h, encoder_outputs)

        # Tile the context so it is concatenated onto every decoder timestep.
        seq_len = tf.shape(x)[1]
        context = tf.expand_dims(context_vector, 1)          # [batch, 1, units]
        context = tf.tile(context, [1, seq_len, 1])          # [batch, seq, units]

        lstm_input = tf.concat([x, context], axis=-1)        # embeddings ++ context
        lstm_output, _, _ = self.lstm(lstm_input, initial_state=[state_h, state_c])
        outputs = self.dense(lstm_output)                    # [batch, seq, vocab]
        return outputs

print("\nBuilding Seq2Seq model WITH attention...")
encoder_attn = Encoder_WithAttention(input_vocab_size, EMBEDDING_DIM, LSTM_UNITS)
decoder_attn = Decoder_WithAttention(target_vocab_size, EMBEDDING_DIM, LSTM_UNITS)
print("✓ Encoder, Attention, and Decoder (WITH attention) created successfully")

### B.4 Training setup and training (with attention) *(provided)*

In [ ]:
optimizer_attn = keras.optimizers.Adam(learning_rate=0.001, clipnorm=1.0)
loss_fn_attn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

def train_step_with_attention(input_batch, target_batch):
    with tf.GradientTape() as tape:
        encoder_outputs, state_h, state_c = encoder_attn(input_batch)
        decoder_input, decoder_target = target_batch[:, :-1], target_batch[:, 1:]
        predictions = decoder_attn(decoder_input, encoder_outputs, [state_h, state_c])
        mask = tf.math.not_equal(decoder_target, 0)
        loss = loss_fn_attn(decoder_target, predictions, sample_weight=mask)
        accuracy = calculate_accuracy(predictions, decoder_target)
    trainable_vars = encoder_attn.trainable_variables + decoder_attn.trainable_variables
    gradients = tape.gradient(loss, trainable_vars)
    optimizer_attn.apply_gradients(zip(gradients, trainable_vars))
    return loss.numpy(), accuracy

def evaluate_with_attention(input_data, target_data):
    total_loss = total_accuracy = 0
    num_batches = len(input_data) // BATCH_SIZE
    for i in range(num_batches):
        input_batch = input_data[i * BATCH_SIZE:(i + 1) * BATCH_SIZE]
        target_batch = target_data[i * BATCH_SIZE:(i + 1) * BATCH_SIZE]
        encoder_outputs, state_h, state_c = encoder_attn(input_batch)
        decoder_input, decoder_target = target_batch[:, :-1], target_batch[:, 1:]
        predictions = decoder_attn(decoder_input, encoder_outputs, [state_h, state_c])
        mask = tf.math.not_equal(decoder_target, 0)
        total_loss += loss_fn_attn(decoder_target, predictions, sample_weight=mask).numpy()
        total_accuracy += calculate_accuracy(predictions, decoder_target)
    return total_loss / num_batches, total_accuracy / num_batches

In [ ]:
print("\n" + "="*70)
print("TRAINING SEQ2SEQ WITH ATTENTION")
print("="*70)

best_val_loss_attn, best_val_acc_attn = float("inf"), 0

for epoch in tqdm(range(EPOCHS)):
    indices = np.random.permutation(len(train_input))
    train_input_shuffled = train_input[indices]
    train_target_shuffled = train_target[indices]

    num_batches = len(train_input) // BATCH_SIZE
    total_loss = total_accuracy = 0
    for i in range(num_batches):
        input_batch = train_input_shuffled[i * BATCH_SIZE:(i + 1) * BATCH_SIZE]
        target_batch = train_target_shuffled[i * BATCH_SIZE:(i + 1) * BATCH_SIZE]
        loss, accuracy = train_step_with_attention(input_batch, target_batch)
        total_loss += loss
        total_accuracy += accuracy

    train_loss, train_accuracy = total_loss / num_batches, total_accuracy / num_batches
    val_loss, val_accuracy = evaluate_with_attention(val_input, val_target)
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {train_loss:.4f}, Acc: {train_accuracy:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.4f}")
    if val_loss < best_val_loss_attn:
        best_val_loss_attn, best_val_acc_attn = val_loss, val_accuracy

print(f"\n✓ Training complete!")
print(f"  Best Validation Loss (WITH ATTENTION): {best_val_loss_attn:.4f}")
print(f"  Best Validation Accuracy (WITH ATTENTION): {best_val_acc_attn:.4f} ({best_val_acc_attn*100:.2f}%)")

## PART C — Comparison and testing *(provided)*

In [ ]:
print(f"\nMetrics Comparison:")
print(f"{'Metric':<30} {'Without Attention':<20} {'With Attention':<20} {'Improvement':<15}")
print("-" * 85)
print(f"{'Validation Loss':<30} {best_val_loss_no_attn:<20.4f} {best_val_loss_attn:<20.4f} "
      f"{((best_val_loss_no_attn - best_val_loss_attn) / best_val_loss_no_attn * 100):>14.2f}%")
print(f"{'Validation Accuracy':<30} {best_val_acc_no_attn:<20.4f} {best_val_acc_attn:<20.4f} "
      f"{((best_val_acc_attn - best_val_acc_no_attn) / max(best_val_acc_no_attn, 1e-9) * 100):>14.2f}%")

In [ ]:
# Greedy word-by-word translation with each model (provided).
def _prep(sentence):
    sentence = preprocess_sentence(sentence)
    inputs = input_tokenizer.texts_to_sequences([sentence])
    return keras.preprocessing.sequence.pad_sequences(inputs, maxlen=MAX_LENGTH, padding="post")

def translate_no_attention(sentence):
    inputs = _prep(sentence)
    encoder_outputs, state_h, state_c = encoder_no_attn(inputs)
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = target_tokenizer.word_index["<start>"]
    decoded, states = [], [state_h, state_c]
    for _ in range(MAX_LENGTH):
        emb = decoder_no_attn.embedding(target_seq)
        out, h, c = decoder_no_attn.lstm(emb, initial_state=states)
        logits = decoder_no_attn.dense(out)
        predicted_id = int(tf.argmax(logits[0, -1]).numpy())
        word = target_tokenizer.index_word.get(predicted_id, "")
        if word == "<end>" or word == "":
            break
        decoded.append(word)
        target_seq = np.array([[predicted_id]])
        states = [h, c]
    return " ".join(decoded)

def translate_with_attention(sentence):
    inputs = _prep(sentence)
    encoder_outputs, state_h, state_c = encoder_attn(inputs)
    context_vector, _ = decoder_attn.attention(state_h, encoder_outputs)
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = target_tokenizer.word_index["<start>"]
    decoded, states = [], [state_h, state_c]
    for _ in range(MAX_LENGTH):
        emb = decoder_attn.embedding(target_seq)                       # [1, 1, emb]
        ctx = tf.expand_dims(context_vector, 1)                        # [1, 1, units]
        out, h, c = decoder_attn.lstm(tf.concat([emb, ctx], axis=-1), initial_state=states)
        logits = decoder_attn.dense(out)
        predicted_id = int(tf.argmax(logits[0, -1]).numpy())
        word = target_tokenizer.index_word.get(predicted_id, "")
        if word == "<end>" or word == "":
            break
        decoded.append(word)
        target_seq = np.array([[predicted_id]])
        states = [h, c]
    return " ".join(decoded)

In [ ]:
print("\n" + "="*70)
print("TRANSLATION EXAMPLES")
print("="*70)

test_cases = [
    {"english": "I am a student.", "german_ground_truth": "ich bin ein student"},
    {"english": "How are you?", "german_ground_truth": "wie geht es dir"},
    {"english": "Good morning.", "german_ground_truth": "guten morgen"},
    {"english": "Thank you.", "german_ground_truth": "danke"},
    {"english": "Where is the station?", "german_ground_truth": "wo ist der bahnhof"},
]

total_bleu_no_attn = total_bleu_attn = 0
for case in test_cases:
    t_no = translate_no_attention(case["english"])
    t_at = translate_with_attention(case["english"])
    b_no = calculate_bleu_score(case["german_ground_truth"], t_no)
    b_at = calculate_bleu_score(case["german_ground_truth"], t_at)
    total_bleu_no_attn += b_no
    total_bleu_attn += b_at
    print(f"\nEN: {case['english']}")
    print(f"  reference       : {case['german_ground_truth']}")
    print(f"  without attention: {t_no!r}  (BLEU {b_no:.2f})")
    print(f"  with attention   : {t_at!r}  (BLEU {b_at:.2f})")

print("\n" + "-"*70)
print(f"Average BLEU without attention: {total_bleu_no_attn/len(test_cases):.3f}")
print(f"Average BLEU with attention   : {total_bleu_attn/len(test_cases):.3f}")

## Checklist before you submit

* [ ] All cells run top-to-bottom without errors, outputs visible.
* [ ] Both models train for all epochs with steadily decreasing loss.
* [ ] The attention model beats the no-attention model on validation loss/accuracy.
* [ ] Translation examples and BLEU scores are shown for both models.
* [ ] In a final markdown cell (2-4 sentences): why does the fixed-size context vector
      become a bottleneck, and how exactly does attention remove it? Bring your
      comparison numbers to the M5 Discussion — and keep them for next module, where
      the same attention idea becomes the full transformer.

**Submit your completed notebook (.ipynb) with all outputs visible.**